> **TrustBreast — Notebook 2.** Table 7; Figs 6–8.

# File 2 — FIXED version (Objective 2: SHAP + LIME)

## How to run
1. Colab → **Runtime → Change runtime type → CPU** (default; no GPU)
2. **Runtime → Run all**; click **Allow** when prompted for Drive access
3. Runtime: approximately **30–45 minutes** (longest step: STEP 7 stable LIME)
4. At the end, share the output of **STEP 9 — RESULTS SUMMARY**.
5. The final cell downloads a **zip file** containing all figures (Fig 6, 7, 8). Share that as well.

## What was fixed
- The old "single-shot LIME" consistency run (which produced an incorrect Fig 8 image) has been **removed**. Only the stable run (15,000 samples × 3) is kept.
- The old "rank 3" label inside the beeswarm now **writes the correct rank automatically**.
- A final summary cell **compares** every number with the previous run and reports ✅/❌ (reproducibility check).

In [ ]:
# Colab: clone the repo (it contains the models/ folder). In local Jupyter this cell does nothing.
import os
if os.path.exists('/content') and not os.path.isdir('models') and not os.path.isdir('../models'):
    !git clone -q https://github.com/Iqra672-ai/TrustBreast.git /content/TrustBreast
    %cd /content/TrustBreast
    !pip -q install -r requirements.txt


## STEP 1 — Determinism + install

In [ ]:
# ============================================
# CELL 0 — DETERMINISM  (run FIRST, before any imports)
# Adds 3 settings that make the DNN give the SAME result on every run:
#   1. os.environ flags   -> GPU/cuDNN deterministic (must be set BEFORE imports)
#   2. all seeds          -> python / numpy / tensorflow
#   3. enable_op_determinism() -> GPU floating-point order LOCK (this was the actual missing piece)
# The reseed() helper later re-fixes the RNG right before the DNN runs.
# ============================================
import os
os.environ['PYTHONHASHSEED']         = '42'
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("enable_op_determinism() ON  ->  DNN now reproducible")
except Exception as e:
    print("Note: enable_op_determinism unavailable (older TF). The remaining fixes still apply.")

def reseed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

print("TF:", tf.__version__, "| Determinism setup done. Now run the remaining cells.")


In [ ]:
!pip -q install scikit-learn xgboost imbalanced-learn tensorflow scipy shap lime dice-ml anthropic matplotlib seaborn

## STEP 2 — Locked model LOAD

In [ ]:
# ===== LOAD the locked 99.12% model (run this FIRST) =====
# Requires the saved model folder in Google Drive: MyDrive/TrustBreast_locked/
# (produced once by File 1 - Objective 1). No retraining here, so the number is always 99.12%.
import os, pickle, joblib, numpy as np, tensorflow as tf
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
# Locked model: first the GitHub repo's models/ folder, otherwise Google Drive
SAVE_DIR = next((p for p in ['models/TrustBreast_locked', '../models/TrustBreast_locked']
                 if os.path.isdir(p)), None)
if SAVE_DIR is None:
    SAVE_DIR = '/content/drive/MyDrive/TrustBreast_locked'
    from google.colab import drive; drive.mount('/content/drive')
print('Loading locked model from:', SAVE_DIR)
rf_model  = joblib.load(SAVE_DIR + '/rf_model.pkl')
xgb_model = joblib.load(SAVE_DIR + '/xgb_model.pkl')
scaler    = joblib.load(SAVE_DIR + '/scaler.pkl')
dnn_best  = tf.keras.models.load_model(SAVE_DIR + '/dnn_best.keras')
dnn_model = tf.keras.models.load_model(SAVE_DIR + '/dnn_model.keras')
with open(SAVE_DIR + '/state.pkl','rb') as f: state = pickle.load(f)
globals().update({k:v for k,v in state.items() if v is not None})
if globals().get('prob_ensemble_val') is None and 'X_val_sc' in globals():
    _rf=rf_model.predict_proba(X_val_sc)[:,1]; _xg=xgb_model.predict_proba(X_val_sc)[:,1]
    _dn=dnn_best.predict(X_val_sc, verbose=0).ravel(); prob_ensemble_val=(_rf+_xg+_dn)/3
model_dnn=dnn_best; rf_aug=rf_model; xgb_aug=xgb_model; feature_names=list(X.columns)
print('LOADED locked model. Ensemble accuracy:', round(accuracy_score(y_test, ens_pred)*100,2), 'percent')

In [ ]:
# --- aliases so O2/O3/O4 cells find the trained models ---
model_dnn = dnn_best        # O3 (MC Dropout) expects this name
rf_aug    = rf_model        # O4 (DiCE) expects this name
xgb_aug   = xgb_model       # O4 (DiCE) expects this name
feature_names = list(X.columns)
print('Bridge ready. Ensemble threshold from O1:', best_ens_thr)

## STEP 3 — SHAP: Random Forest + XGBoost (TreeExplainer)

In [ ]:
# ============================================================
# OBJECTIVE 2 — SHAP TreeExplainer
# Random Forest + XGBoost — Individual Analysis + Plots
# Methodology ref: O2 Steps 2.1, 2.4, 2.5
# ============================================================

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

try:
    import shap
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'shap', '-q'])
    import shap

# ── Publication style ────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

FEATURE_NAMES = list(X.columns)
N_TEST        = int(X_test_sc.shape[0])   # 114
N_FEAT        = int(X_test_sc.shape[1])   # 30

print("=" * 62)
print("  SHAP TreeExplainer — RF + XGBoost")
print("=" * 62)
print(f"  Test patients : {N_TEST}")
print(f"  Features      : {N_FEAT}")

# ─────────────────────────────────────────────────────────────
# STEP A — RANDOM FOREST TreeExplainer
# ─────────────────────────────────────────────────────────────
# WHY TreeExplainer:
#   RF is a collection of decision trees.
#   TreeExplainer walks every tree and
#   computes exactly how much each feature
#   contributed — no approximation.
#
# WHAT IT RETURNS:
#   For binary RF: list of 2 arrays
#   [class_0_shap, class_1_shap]
#   We take class_1 = Malignant class
# ─────────────────────────────────────────────────────────────

print()
print("─" * 62)
print("  STEP A — Random Forest SHAP")
print("─" * 62)
print("  Building TreeExplainer ...")

rf_explainer = shap.TreeExplainer(rf_model)

print("  Computing SHAP values for 114 patients ...")
print("  (500 trees — may take 30-60 seconds)")

rf_shap_raw = rf_explainer.shap_values(X_test_sc)

# ── Handle shape ─────────────────────────────────────────────
if isinstance(rf_shap_raw, list):
    shap_rf = np.array(rf_shap_raw[1])
    print("  Format: list → class-1 (Malignant) liya")
elif isinstance(rf_shap_raw, np.ndarray) and rf_shap_raw.ndim == 3:
    shap_rf = rf_shap_raw[:, :, 1]
    print("  Format: 3D array → class-1 slice liya")
else:
    shap_rf = np.array(rf_shap_raw)
    print("  Format: 2D array directly")

assert shap_rf.shape == (N_TEST, N_FEAT), \
    f"Shape error: {shap_rf.shape}"
print(f"  shap_rf shape : {shap_rf.shape}  ✅")

# ── RF Global Stats ──────────────────────────────────────────
mean_abs_rf = np.abs(shap_rf).mean(axis=0)
rf_rank     = np.argsort(mean_abs_rf)[::-1]

print()
print("  RF — Top 10 Features (by mean |SHAP|):")
print(f"  {'Rank':<5} {'Feature':<32} {'Mean|SHAP|':>10}  Direction")
print(f"  {'-'*60}")
for r in range(10):
    idx       = rf_rank[r]
    feat      = FEATURE_NAMES[idx]
    val       = mean_abs_rf[idx]
    # Direction: positive mean = pushes toward Malignant
    mean_sign = np.mean(shap_rf[:, idx])
    dirn      = "→ Malignant" if mean_sign > 0 else "→ Benign"
    print(f"  {r+1:<5} {feat:<32} {val:>10.4f}  {dirn}")

print(f"\n  RF SHAP value range : "
      f"[{shap_rf.min():.4f},  {shap_rf.max():.4f}]")
print(f"  (Probability space — values between -1 and +1)")
print(f"  ✅  RF SHAP complete")

# ─────────────────────────────────────────────────────────────
# STEP B — XGBOOST TreeExplainer
# ─────────────────────────────────────────────────────────────
# WHY SAME APPROACH:
#   XGBoost is also tree-based — TreeExplainer works the
#   same way. One difference, however:
#
# IMPORTANT SCALE DIFFERENCE:
#   RF SHAP   → probability space  → values ~±0.07
#   XGB SHAP  → log-odds space     → values ~±0.97
#
#   Both rank the same features as important, but the
#   magnitudes differ. Hence normalization in the ensemble
#   is required (already done in Step D).
# ─────────────────────────────────────────────────────────────

print()
print("─" * 62)
print("  STEP B — XGBoost SHAP")
print("─" * 62)
print("  Building TreeExplainer ...")

xgb_explainer = shap.TreeExplainer(xgb_model)

print("  Computing SHAP values for 114 patients ...")
print("  (XGBoost is fast — 5-15 seconds)")

xgb_shap_raw = xgb_explainer.shap_values(X_test_sc)

# ── Handle shape ─────────────────────────────────────────────
if isinstance(xgb_shap_raw, list):
    shap_xgb = np.array(xgb_shap_raw[1])
    print("  Format: list → class-1 liya")
elif isinstance(xgb_shap_raw, np.ndarray) and xgb_shap_raw.ndim == 3:
    shap_xgb = xgb_shap_raw[:, :, 1]
    print("  Format: 3D array → class-1 slice")
else:
    shap_xgb = np.array(xgb_shap_raw)
    print("  Format: 2D array directly")

assert shap_xgb.shape == (N_TEST, N_FEAT), \
    f"Shape error: {shap_xgb.shape}"
print(f"  shap_xgb shape : {shap_xgb.shape}  ✅")

# ── XGB Global Stats ─────────────────────────────────────────
mean_abs_xgb = np.abs(shap_xgb).mean(axis=0)
xgb_rank     = np.argsort(mean_abs_xgb)[::-1]

print()
print("  XGBoost — Top 10 Features (by mean |SHAP|):")
print(f"  {'Rank':<5} {'Feature':<32} {'Mean|SHAP|':>10}  Direction")
print(f"  {'-'*60}")
for r in range(10):
    idx       = xgb_rank[r]
    feat      = FEATURE_NAMES[idx]
    val       = mean_abs_xgb[idx]
    mean_sign = np.mean(shap_xgb[:, idx])
    dirn      = "→ Malignant" if mean_sign > 0 else "→ Benign"
    print(f"  {r+1:<5} {feat:<32} {val:>10.4f}  {dirn}")

print(f"\n  XGB SHAP value range : "
      f"[{shap_xgb.min():.4f},  {shap_xgb.max():.4f}]")
print(f"  (Log-odds space — values much larger than RF)")
print(f"  ✅  XGB SHAP complete")

# ─────────────────────────────────────────────────────────────
# RF vs XGB AGREEMENT TABLE
# ─────────────────────────────────────────────────────────────
print()
print("=" * 62)
print("  RF vs XGBoost — Top 10 Feature Agreement")
print("=" * 62)
print(f"  {'Rank':<5} {'RF Feature':<30} {'XGB Feature':<30} {'Match?'}")
print(f"  {'-'*72}")

matches = 0
for r in range(10):
    rf_f  = FEATURE_NAMES[rf_rank[r]]
    xgb_f = FEATURE_NAMES[xgb_rank[r]]
    match = "✅ SAME" if rf_f == xgb_f else "—  different"
    if rf_f == xgb_f:
        matches += 1
    print(f"  {r+1:<5} {rf_f:<30} {xgb_f:<30} {match}")

print(f"\n  Exact matches (top 10) : {matches}/10")
print(f"  Top-4 agreement        : "
      f"{'✅ Strong' if matches >= 4 else '⚠ Check'}")

# ─────────────────────────────────────────────────────────────
# PLOT 1 — RF SHAP Beeswarm
# ─────────────────────────────────────────────────────────────
print()
print("=" * 62)
print("  PLOT 1 — RF SHAP Beeswarm")
print("=" * 62)

plt.close('all')

shap.summary_plot(
    shap_rf,
    X_test_sc,
    feature_names = FEATURE_NAMES,
    max_display   = 15,
    show          = False,
    plot_size     = (10, 8),
)

fig1       = plt.gcf()
ax1        = fig1.axes[0]

ax1.set_xlabel("SHAP Value  (← Benign  |  Malignant →)  [Probability Space]",
               fontsize=10)
ax1.set_title(
    "Random Forest — SHAP Beeswarm Plot\n"
    "TrustBreast | WBCD (114 test patients)",
    fontsize=12, fontweight='bold', pad=14
)

if len(fig1.axes) > 1:
    fig1.axes[1].set_ylabel('Feature Value\n(0=Low  1=High)',
                            fontsize=8)

fig1.tight_layout()
fig1.savefig('RF_SHAP_Beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()
print("  ✅  RF_SHAP_Beeswarm.png saved")

# ─────────────────────────────────────────────────────────────
# PLOT 2 — XGBoost SHAP Beeswarm
# ─────────────────────────────────────────────────────────────
print()
print("=" * 62)
print("  PLOT 2 — XGBoost SHAP Beeswarm")
print("=" * 62)

plt.close('all')

shap.summary_plot(
    shap_xgb,
    X_test_sc,
    feature_names = FEATURE_NAMES,
    max_display   = 15,
    show          = False,
    plot_size     = (10, 8),
)

fig2 = plt.gcf()
ax2  = fig2.axes[0]

ax2.set_xlabel("SHAP Value  (← Benign  |  Malignant →)  [Log-Odds Space]",
               fontsize=10)
ax2.set_title(
    "XGBoost — SHAP Beeswarm Plot\n"
    "TrustBreast | WBCD (114 test patients)",
    fontsize=12, fontweight='bold', pad=14
)

if len(fig2.axes) > 1:
    fig2.axes[1].set_ylabel('Feature Value\n(0=Low  1=High)',
                            fontsize=8)

fig2.tight_layout()
fig2.savefig('XGB_SHAP_Beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()
print("  ✅  XGB_SHAP_Beeswarm.png saved")

# ─────────────────────────────────────────────────────────────
# PLOT 3 — RF vs XGB Side-by-Side Bar Comparison
# ─────────────────────────────────────────────────────────────
print()
print("=" * 62)
print("  PLOT 3 — RF vs XGB Feature Importance Comparison")
print("=" * 62)

plt.close('all')

# Top 10 features from RF ranking
top10_idx  = rf_rank[:10]
top10_feat = [FEATURE_NAMES[i] for i in top10_idx]

# Normalize both to 0-1 scale for fair visual comparison
rf_vals_norm  = mean_abs_rf[top10_idx]
rf_vals_norm  = rf_vals_norm / rf_vals_norm.max()

xgb_vals_raw  = mean_abs_xgb[top10_idx]
xgb_vals_norm = xgb_vals_raw / xgb_vals_raw.max()

x      = np.arange(len(top10_feat))
width  = 0.38

fig3, ax3 = plt.subplots(figsize=(13, 6))

bars_rf  = ax3.bar(x - width/2, rf_vals_norm,
                   width, label='Random Forest',
                   color='#2980B9', alpha=0.85,
                   edgecolor='white', linewidth=0.5)

bars_xgb = ax3.bar(x + width/2, xgb_vals_norm,
                   width, label='XGBoost',
                   color='#E67E22', alpha=0.85,
                   edgecolor='white', linewidth=0.5)

# Value labels on bars
for bar in bars_rf:
    h = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2,
             h + 0.01, f'{h:.2f}',
             ha='center', va='bottom',
             fontsize=7.5, color='#2980B9',
             fontweight='bold')

for bar in bars_xgb:
    h = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2,
             h + 0.01, f'{h:.2f}',
             ha='center', va='bottom',
             fontsize=7.5, color='#C0392B',
             fontweight='bold')

ax3.set_xticks(x)
ax3.set_xticklabels(top10_feat, rotation=35,
                    ha='right', fontsize=9)
ax3.set_ylabel('Normalised Mean |SHAP|  (0 → 1)', fontsize=10)
ax3.set_title(
    "RF vs XGBoost — SHAP Feature Importance Comparison\n"
    "Both normalised to [0,1] for fair visual comparison  "
    "| TrustBreast | WBCD",
    fontsize=11, fontweight='bold', pad=14
)
ax3.legend(fontsize=10, loc='upper right')
ax3.set_ylim(0, 1.18)
ax3.grid(axis='y', alpha=0.3, linestyle='--')
ax3.set_xlim(-0.6, len(top10_feat) - 0.4)

fig3.tight_layout()
fig3.savefig('RF_XGB_SHAP_Comparison.png',
             dpi=300, bbox_inches='tight')
plt.show()
print("  ✅  RF_XGB_SHAP_Comparison.png saved")

# ─────────────────────────────────────────────────────────────
# COMPLETE SUMMARY TABLE
# ─────────────────────────────────────────────────────────────
print()
print("=" * 62)
print("  COMPLETE SUMMARY — RF + XGB SHAP")
print("=" * 62)
print(f"\n  {'Rank':<5} {'Feature':<32} "
      f"{'RF |SHAP|':>10} {'XGB |SHAP|':>11}  "
      f"{'RF Rank':>8} {'XGB Rank':>9}")
print(f"  {'-'*80}")

# Build XGB rank lookup
xgb_rank_lookup = {idx: r+1 for r, idx in enumerate(xgb_rank)}

for r in range(15):
    idx       = rf_rank[r]
    feat      = FEATURE_NAMES[idx]
    rf_val    = mean_abs_rf[idx]
    xgb_val   = mean_abs_xgb[idx]
    xgb_r     = xgb_rank_lookup[idx]
    same      = "✅" if abs(r+1 - xgb_r) <= 1 else ""
    print(f"  {r+1:<5} {feat:<32} "
          f"{rf_val:>10.4f} {xgb_val:>11.4f}  "
          f"{r+1:>8} {xgb_r:>9}  {same}")

print(f"""
  Scale Note:
    RF  values → Probability space  (small numbers ~0.07)
    XGB values → Log-odds space     (large numbers ~0.97)
    Same features, different scales — normalized in ensemble.

  Files saved:
    ✅  RF_SHAP_Beeswarm.png
    ✅  XGB_SHAP_Beeswarm.png
    ✅  RF_XGB_SHAP_Comparison.png

  ─────────────────────────────────────────────────────
  Paste full output + share 3 plots.
  Each result will then be reviewed in detail.
  ─────────────────────────────────────────────────────
""")

## STEP 4 — SHAP: DNN (DeepExplainer)

In [ ]:
import numpy as np
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("  STEP 2.2 — DNN SHAP (DeepExplainer + Beeswarm)")
print("="*60)

FEATURE_NAMES = list(X.columns)

# ── Background sample (100 from SMOTE train, seed=42) ────────
np.random.seed(42)

# FIX: Define X_train_sm using SMOTE as indicated by the comments
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train) # Assuming X_train_sc and y_train are available and scaled/preprocessed

bg_idx = np.random.choice(X_train_sm.shape[0], 100, replace=False)
background = (X_train_sm.values[bg_idx]
             if hasattr(X_train_sm, 'values')
             else X_train_sm[bg_idx])

print(f"  Background : {background.shape}  (100 samples)")

# ── Test data as array ───────────────────────────────────────
X_test_arr = (X_test_sc.values
              if hasattr(X_test_sc, 'values')
              else np.asarray(X_test_sc))
print(f"  Test data  : {X_test_arr.shape}")

# ── DeepExplainer ────────────────────────────────────────────
print("\n  Running DeepExplainer (~30-60 sec)...")

explainer_dnn = shap.DeepExplainer(dnn_best, background)
shap_dnn_raw  = explainer_dnn.shap_values(X_test_arr)

# DeepExplainer often returns a list or (n,30,1) — clean it
if isinstance(shap_dnn_raw, list):
    shap_dnn = np.array(shap_dnn_raw[0])
else:
    shap_dnn = np.array(shap_dnn_raw)

# Squeeze trailing dim if (114,30,1) → (114,30)
if shap_dnn.ndim == 3 and shap_dnn.shape[-1] == 1:
    shap_dnn = shap_dnn[:, :, 0]

print(f"\n  ✅ shap_dnn shape : {shap_dnn.shape}")
print(f"     Range : [{shap_dnn.min():.4f}, {shap_dnn.max():.4f}]")
print(f"     Std   : {shap_dnn.std():.5f}")

# ── DNN Top 5 ────────────────────────────────────────────────
mean_abs_dnn = np.abs(shap_dnn).mean(axis=0)
dnn_rank_idx = np.argsort(mean_abs_dnn)[::-1]

print(f"\n  DNN Top 5 features:")
for r in range(5):
    idx = dnn_rank_idx[r]
    print(f"    {r+1}. {FEATURE_NAMES[idx]:<28} {mean_abs_dnn[idx]:.4f}")

In [ ]:
# ── Beeswarm plot (SHAP built-in summary_plot) ───────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size'  : 11,
})

# Build SHAP Explanation object for clean beeswarm
shap_exp_dnn = shap.Explanation(
    values        = shap_dnn,
    data          = X_test_arr,
    feature_names = FEATURE_NAMES
)

plt.figure(figsize=(11, 10))

shap.plots.beeswarm(
    shap_exp_dnn,
    max_display = 20,
    show        = False,
    color_bar   = True
)

# ── Style: top-10 features red+bold ──────────────────────────
top10_set = {
    'concave points_worst','perimeter_worst','concave points_mean',
    'radius_worst','area_worst','perimeter_mean',
    'radius_mean','area_mean','concavity_mean','concavity_worst'
}
def norm(s): return s.strip().lower().replace(' ','_')
top10_norm = {norm(f) for f in top10_set}

ax = plt.gca()
for lbl in ax.get_yticklabels():
    fname = lbl.get_text()
    if norm(fname) in top10_norm:
        lbl.set_color('crimson')
        lbl.set_fontweight('bold')

plt.title('DNN SHAP Beeswarm — Feature Impact on Malignant Prediction\n'
          'TrustBreast | WBCD | Deep Explainer (Gradient space)',
          fontsize=12, fontweight='bold', pad=14)
plt.xlabel('SHAP value (impact on model output)', fontsize=11)

# Annotation box
ax.text(0.02, 0.02,
        "DeepExplainer | n=114 test patients\n"
        "Background = 100 SMOTE samples\n"
        "Red = statistical Top-10 feature",
        transform=ax.transAxes, fontsize=8.5,
        va='bottom', ha='left',
        bbox=dict(boxstyle='round,pad=0.4',
                  facecolor='lightyellow', alpha=0.85))

plt.tight_layout()
for ext in ['png','pdf']:
    plt.savefig(f'DNN_SHAP_Beeswarm_FINAL.{ext}',
                dpi=300, bbox_inches='tight')
print("  ✅ Saved → DNN_SHAP_Beeswarm_FINAL.png / .pdf")
plt.show()

## STEP 5 — Ensemble SHAP (each model / global std) + Figures

In [ ]:
# ============================================================
# OBJECTIVE 2 — STEP 2.3
# Ensemble SHAP Aggregate (Normalized Average)
# Formula: (shap_rf/std_rf + shap_xgb/std_xgb +
#           shap_dnn/std_dnn) / 3
# ============================================================

import numpy as np
import warnings
warnings.filterwarnings('ignore')

FEATURE_NAMES = list(X.columns)
N_TEST, N_FEAT = shap_rf.shape

print("="*62)
print("  STEP 2.3 — Ensemble SHAP Aggregate")
print("="*62)

# ── Verify all three SHAP arrays are ready ───────────────────────
for name, arr in [('shap_rf', shap_rf),
                  ('shap_xgb', shap_xgb),
                  ('shap_dnn', shap_dnn)]:
    assert arr.shape == (N_TEST, N_FEAT), f"{name} shape mismatch!"
    print(f"  {name:<10} : {arr.shape}  ✅")

# ─────────────────────────────────────────────────────────────
# STEP 1 — Global std of each model (normalization factor)
# ─────────────────────────────────────────────────────────────
std_rf  = float(np.std(shap_rf))
std_xgb = float(np.std(shap_xgb))
std_dnn = float(np.std(shap_dnn))

print()
print("─"*62)
print("  STEP 1 — Normalization Factors (global std)")
print("─"*62)
print(f"  std_rf  : {std_rf:.5f}")
print(f"  std_xgb : {std_xgb:.5f}")
print(f"  std_dnn : {std_dnn:.5f}")
print(f"  XGB scale = {std_xgb/std_rf:.1f}x bigger than RF")

# ─────────────────────────────────────────────────────────────
# STEP 2 — First show what happens WITHOUT normalization
# ─────────────────────────────────────────────────────────────
raw_rf_contrib  = np.abs(shap_rf).mean()
raw_xgb_contrib = np.abs(shap_xgb).mean()
raw_dnn_contrib = np.abs(shap_dnn).mean()
raw_total = raw_rf_contrib + raw_xgb_contrib + raw_dnn_contrib

print()
print("─"*62)
print("  STEP 2 — WITHOUT Normalization (showing the problem)")
print("─"*62)
print(f"  RF  contribution : {raw_rf_contrib/raw_total*100:5.1f}%")
print(f"  XGB contribution : {raw_xgb_contrib/raw_total*100:5.1f}%  ← hijack!")
print(f"  DNN contribution : {raw_dnn_contrib/raw_total*100:5.1f}%")
print("  → XGB alone dominates — biased ensemble")

# ─────────────────────────────────────────────────────────────
# STEP 3 — Normalize, then aggregate
# ─────────────────────────────────────────────────────────────
shap_rf_norm  = shap_rf  / std_rf
shap_xgb_norm = shap_xgb / std_xgb
shap_dnn_norm = shap_dnn / std_dnn

shap_ensemble = (shap_rf_norm + shap_xgb_norm + shap_dnn_norm) / 3.0

print()
print("─"*62)
print("  STEP 3 — AFTER Normalization (fair contribution)")
print("─"*62)
norm_rf  = np.abs(shap_rf_norm).mean()
norm_xgb = np.abs(shap_xgb_norm).mean()
norm_dnn = np.abs(shap_dnn_norm).mean()
norm_tot = norm_rf + norm_xgb + norm_dnn
print(f"  RF  contribution : {norm_rf/norm_tot*100:5.1f}%")
print(f"  XGB contribution : {norm_xgb/norm_tot*100:5.1f}%")
print(f"  DNN contribution : {norm_dnn/norm_tot*100:5.1f}%")
print("  → All three now balanced (~33% each)  ✅")

print()
print(f"  shap_ensemble shape : {shap_ensemble.shape}")
print(f"  Range : [{shap_ensemble.min():.4f}, {shap_ensemble.max():.4f}]")

# ─────────────────────────────────────────────────────────────
# STEP 4 — Ensemble Top 10 + rank lookups (for Step 2.5)
# ─────────────────────────────────────────────────────────────
mean_abs_ens = np.abs(shap_ensemble).mean(axis=0)
ens_rank_idx = np.argsort(mean_abs_ens)[::-1]

# Also build per-model rank indices (needed in Step 2.5)
rf_rank_idx  = np.argsort(np.abs(shap_rf).mean(axis=0))[::-1]
xgb_rank_idx = np.argsort(np.abs(shap_xgb).mean(axis=0))[::-1]
dnn_rank_idx = np.argsort(np.abs(shap_dnn).mean(axis=0))[::-1]

# Statistical top-10 set (normalized names)
def norm_name(s): return s.strip().lower().replace(' ','_')
top10_norm = {norm_name(f) for f in [
    'concave points_worst','perimeter_worst','concave points_mean',
    'radius_worst','area_worst','perimeter_mean',
    'radius_mean','area_mean','concavity_mean','concavity_worst'
]}

print()
print("="*62)
print("  Ensemble Top 10 Features")
print("="*62)
print(f"  {'Rank':<5} {'Feature':<30} {'Ens|SHAP|':>10}  {'In Top-10?'}")
print(f"  {'-'*60}")
for r in range(10):
    idx  = ens_rank_idx[r]
    feat = FEATURE_NAMES[idx]
    val  = mean_abs_ens[idx]
    in10 = "✅ YES" if norm_name(feat) in top10_norm else "⭐ cross-method"
    print(f"  {r+1:<5} {feat:<30} {val:>10.4f}  {in10}")

# ─────────────────────────────────────────────────────────────
# FINAL — Variables ready
# ─────────────────────────────────────────────────────────────
print()
print("="*62)
print("  STEP 2.3 COMPLETE — Variables ready for Step 2.5")
print("="*62)
print(f"""
  ✅  shap_ensemble  {shap_ensemble.shape}   (normalized avg)
  ✅  mean_abs_ens   (30,)   global importance
  ✅  ens_rank_idx   ranked feature indices
  ✅  rf_rank_idx / xgb_rank_idx / dnn_rank_idx
  ✅  top10_norm     (statistical top-10 set)
  ✅  std_rf={std_rf:.4f}  std_xgb={std_xgb:.4f}  std_dnn={std_dnn:.4f}

  NEXT → Step 2.5 (Top 5 Identification + clinical meaning)
""")

In [ ]:
# ============================================================
# OBJECTIVE 2 — STEP 2.4
# Global SHAP Visualizations
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import shap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

FEATURE_NAMES = list(X.columns)
X_test_arr = (X_test_sc.values if hasattr(X_test_sc, 'values')
              else np.asarray(X_test_sc))

def norm_name(s): return s.strip().lower().replace(' ','_')
top10_norm_set = {norm_name(f) for f in [
    'concave points_worst','perimeter_worst','concave points_mean',
    'radius_worst','area_worst','perimeter_mean',
    'radius_mean','area_mean','concavity_mean','concavity_worst'
]}
def is_top10(f): return norm_name(f) in top10_norm_set
def lbl_style(f):
    return ('crimson','bold') if is_top10(f) else ('black','normal')

print("✅ Config ready —", len(FEATURE_NAMES), "features")

In [ ]:
# ── Figure 1: Mean |SHAP| bar — 3 models side by side ────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

configs = [
    (shap_rf,  'Random Forest', '#2980B9', 'Probability'),
    (shap_xgb, 'XGBoost',       '#E67E22', 'Log-odds'),
    (shap_dnn, 'DNN',           '#8E44AD', 'Gradient'),
]

for ax, (sv, mname, color, space) in zip(axes, configs):
    mabs  = np.abs(sv).mean(axis=0)
    order = np.argsort(mabs)[::-1][:15]
    feats = [FEATURE_NAMES[i] for i in order]
    vals  = mabs[order]

    bcolors = ['crimson' if is_top10(f) else color for f in feats]
    ax.barh(range(len(feats)), vals[::-1],
            color=bcolors[::-1], edgecolor='white',
            linewidth=0.5, height=0.7)

    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels(feats[::-1], fontsize=9)
    for lab, f in zip(ax.get_yticklabels(), feats[::-1]):
        c, w = lbl_style(f); lab.set_color(c); lab.set_fontweight(w)

    ax.set_xlabel(f'Mean |SHAP|  ({space} space)', fontsize=10)
    ax.set_title(f'{mname}', fontsize=12, fontweight='bold', pad=10)
    ax.grid(axis='x', alpha=0.3, linestyle='--')

red_p = mpatches.Patch(color='crimson', label='Statistical Top-10')
oth_p = mpatches.Patch(color='#888888', label='Other features')
fig.legend(handles=[red_p, oth_p], loc='lower center', ncol=2,
           fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.03))

plt.suptitle('Global SHAP Feature Importance — Per Model (Mean |SHAP|)\n'
             'TrustBreast | WBCD | 114 test patients',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
for ext in ['png','pdf']:
    plt.savefig(f'Fig_Global_SHAP_Bar_PerModel.{ext}',
                dpi=300, bbox_inches='tight')
print("✅ Saved → Fig_Global_SHAP_Bar_PerModel.png / .pdf")
plt.show()

In [ ]:
# ── Figure 2: Ensemble SHAP Beeswarm ─────────────────────────
shap_exp_ens = shap.Explanation(
    values        = shap_ensemble,
    data          = X_test_arr,
    feature_names = FEATURE_NAMES
)

plt.figure(figsize=(11, 10))
shap.plots.beeswarm(shap_exp_ens, max_display=20,
                    show=False, color_bar=True)

ax = plt.gca()
for lbl in ax.get_yticklabels():
    f = lbl.get_text()
    if is_top10(f):
        lbl.set_color('crimson'); lbl.set_fontweight('bold')

plt.title('Ensemble SHAP Beeswarm — Normalized Average (RF + XGB + DNN)\n'
          'TrustBreast | WBCD | Impact on Malignant Prediction',
          fontsize=12, fontweight='bold', pad=14)
plt.xlabel('Ensemble SHAP value (normalized)', fontsize=11)

_tw = list(ens_rank_idx).index(FEATURE_NAMES.index('texture_worst')) + 1
ax.text(0.02, 0.02,
        "Normalized avg of 3 models (each / global SHAP std)\n"
        "n=114 patients | Red = top-10 by Pearson correlation\n"
        f"texture_worst = ensemble SHAP rank {_tw}",
        transform=ax.transAxes, fontsize=8.5, va='bottom', ha='left',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.85))

plt.tight_layout()
for ext in ['png','pdf']:
    plt.savefig(f'Fig_Ensemble_SHAP_Beeswarm.{ext}',
                dpi=300, bbox_inches='tight')
print("✅ Saved → Fig_Ensemble_SHAP_Beeswarm.png / .pdf")
plt.show()

In [ ]:
# ── Figure 3: Top-10 rank across RF/XGB/DNN/Ensemble ─────────
top10_idx  = ens_rank_idx[:10]
top10_feat = [FEATURE_NAMES[i] for i in top10_idx]

def rank_in(rank_idx_arr, fidx):
    pos = np.where(rank_idx_arr == fidx)[0]
    return int(pos[0]) + 1 if len(pos) else 31

ranks = {
    'RF'      : [rank_in(rf_rank_idx,  i) for i in top10_idx],
    'XGB'     : [rank_in(xgb_rank_idx, i) for i in top10_idx],
    'DNN'     : [rank_in(dnn_rank_idx, i) for i in top10_idx],
    'Ensemble': list(range(1, 11)),
}
colors = ['#2980B9','#E67E22','#8E44AD','#27AE60']

x, width = np.arange(10), 0.2
fig, ax = plt.subplots(figsize=(16, 6))
for i, (m, rv) in enumerate(ranks.items()):
    ax.bar(x + (i-1.5)*width, rv, width, label=m,
           color=colors[i], edgecolor='white', linewidth=0.5, alpha=0.85)

short = [f.replace('concave_points','cc_pts').replace('perimeter','peri')
          .replace('concavity','conc').replace('_worst','_W')
          .replace('_mean','_M').replace('radius','rad') for f in top10_feat]
ax.set_xticks(x); ax.set_xticklabels(short, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Feature Rank (lower = more important)', fontsize=11)
ax.set_title('Cross-Model SHAP Rank Comparison — Top-10 Ensemble Features\n'
             'TrustBreast | WBCD', fontsize=13, fontweight='bold', pad=12)
ax.invert_yaxis(); ax.set_ylim(16, 0)
ax.axhline(10, color='gray', ls=':', lw=0.8, alpha=0.6)
ax.grid(axis='y', alpha=0.25, linestyle='--')
ax.legend(loc='lower right', fontsize=10, frameon=True)

# consensus shading (top-5 in all 3 base models)
for xi, fidx in enumerate(top10_idx):
    if all(rank_in(r, fidx) <= 5 for r in [rf_rank_idx, xgb_rank_idx, dnn_rank_idx]):
        ax.axvspan(xi-0.45, xi+0.45, alpha=0.08, color='gold', zorder=0)

ax.text(0.01, 0.06, "Gold = consensus top-5 across all 3 base models",
        transform=ax.transAxes, fontsize=8.5, va='bottom',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.85))

plt.tight_layout()
for ext in ['png','pdf']:
    plt.savefig(f'Fig_CrossModel_Rank_Comparison.{ext}',
                dpi=300, bbox_inches='tight')
print("✅ Saved → Fig_CrossModel_Rank_Comparison.png / .pdf")
plt.show()

In [ ]:
# ── Figure 4: Heatmap (top-10 features × 114 patients) ───────
y_test_arr = np.array(y_test).ravel()
prob_mal   = np.array(prob_ensemble)[:, 1] if np.ndim(prob_ensemble) == 2 \
             else np.array(prob_ensemble).ravel()

top10_idx  = ens_rank_idx[:10]
top10_feat = [FEATURE_NAMES[i] for i in top10_idx]
heat = shap_ensemble[:, top10_idx]

# sort: malignant first, then by confidence
order = np.argsort(-(y_test_arr * 10 + prob_mal))
heat_s = heat[order]; lbl_s = y_test_arr[order]
n_mal = int(lbl_s.sum())

fig, ax = plt.subplots(figsize=(13, 8))
vmax = np.abs(heat_s).max()
im = ax.imshow(heat_s.T, cmap='RdBu_r', aspect='auto',
               vmin=-vmax, vmax=vmax, interpolation='nearest')

ax.axvline(n_mal - 0.5, color='black', lw=1.5, ls='--', alpha=0.8)
ax.set_xticks([n_mal/2, n_mal + (114-n_mal)/2])
ax.set_xticklabels([f'Malignant (n={n_mal})', f'Benign (n={114-n_mal})'],
                   fontsize=10)
ax.set_xlabel('Test Patients (sorted: Malignant | Benign)', fontsize=11)

ax.set_yticks(range(10)); ax.set_yticklabels(top10_feat, fontsize=9)
for lab, f in zip(ax.get_yticklabels(), top10_feat):
    c, w = lbl_style(f); lab.set_color(c); lab.set_fontweight(w)

cb = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cb.set_label('Ensemble SHAP\n← Benign  |  Malignant →', fontsize=9)

ax.set_title('Ensemble SHAP Heatmap — Top-10 Features × 114 Patients\n'
             'TrustBreast | WBCD', fontsize=13, fontweight='bold', pad=12)
ax.text(0.0, -0.1, "Red = pushes Malignant  |  Blue = pushes Benign",
        transform=ax.transAxes, fontsize=8.5,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.85))

plt.tight_layout()
for ext in ['png','pdf']:
    plt.savefig(f'Fig_SHAP_Heatmap_Ensemble.{ext}',
                dpi=300, bbox_inches='tight')
print("✅ Saved → Fig_SHAP_Heatmap_Ensemble.png / .pdf")
plt.show()

print("\n" + "="*55)
print("  STEP 2.4 COMPLETE — 4 global figures saved")
print("="*55)
print("  1. Fig_Global_SHAP_Bar_PerModel")
print("  2. Fig_Ensemble_SHAP_Beeswarm")
print("  3. Fig_CrossModel_Rank_Comparison")
print("  4. Fig_SHAP_Heatmap_Ensemble")
print("\n  NEXT → Step 2.5 (Top 5 Identification)")

## STEP 6 — Top-5 features (Fig 8)

In [ ]:
# ============================================================
# OBJECTIVE 2 — STEP 2.5
# Top 5 Feature Identification + Clinical Meaning
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})

FEATURE_NAMES = list(X.columns)
y_test_arr = np.array(y_test).ravel()
X_test_arr = (X_test_sc.values if hasattr(X_test_sc, 'values')
              else np.asarray(X_test_sc))

# ── Rank lookups (from Step 2.3 variables) ───────────────────
def rank_lookup(rank_idx):
    return {int(rank_idx[r]): r+1 for r in range(len(rank_idx))}
rf_rnk  = rank_lookup(rf_rank_idx)
xgb_rnk = rank_lookup(xgb_rank_idx)
dnn_rnk = rank_lookup(dnn_rank_idx)

# Statistical top-10 order (for comparison)
STAT_TOP10 = ['concave points_worst','perimeter_worst','concave points_mean',
              'radius_worst','area_worst','perimeter_mean','radius_mean',
              'area_mean','concavity_mean','concavity_worst']
def nm(s): return s.strip().lower().replace(' ','_')
stat_rank = {nm(f): i+1 for i, f in enumerate(STAT_TOP10)}

print("="*62)
print("  STEP 2.5 — Top 5 Feature Identification")
print("="*62)

# ─────────────────────────────────────────────────────────────
# PART 1 — Top 5 Formally Identified
# ─────────────────────────────────────────────────────────────
top5_idx  = [int(ens_rank_idx[r]) for r in range(5)]
top5_name = [FEATURE_NAMES[i] for i in top5_idx]
top5_val  = [float(mean_abs_ens[i]) for i in top5_idx]

print(f"\n  {'Rank':<5}{'Feature':<26}{'Ens|SHAP|':>10}"
      f"{'RF':>5}{'XGB':>5}{'DNN':>5}  Status")
print(f"  {'-'*68}")
for r, (idx, f, v) in enumerate(zip(top5_idx, top5_name, top5_val), 1):
    in10 = "Confirmed" if nm(f) in {nm(x) for x in STAT_TOP10} else "cross-method ⭐"
    print(f"  {r:<5}{f:<26}{v:>10.4f}"
          f"{rf_rnk[idx]:>5}{xgb_rnk[idx]:>5}{dnn_rnk[idx]:>5}  {in10}")

# ─────────────────────────────────────────────────────────────
# PART 2 — Clinical Meaning
# ─────────────────────────────────────────────────────────────
clinical = {
    'perimeter_worst': ('Boundary length of the largest cell',
        'Large irregular boundary = aggressive cancer growth',
        'perimeter_worst captures the most extreme cell boundary, '
        'reflecting aggressive morphological change in malignancy.'),
    'area_worst': ('Nuclear area of the largest cell',
        'Large nucleus = primary cancer sign (cytology)',
        'area_worst represents the largest nuclear area, consistent '
        'with cytological criteria for malignancy.'),
    'concave_points_worst': ('Most indentations in a single cell',
        'Jagged irregular boundary = malignant hallmark',
        'concave_points_worst reflects severity of boundary irregularity, '
        'a hallmark of malignant nuclear morphology.'),
    'concave_points_mean': ('Average indentations across all cells',
        'Consistently irregular = malignant tumour',
        'concave_points_mean quantifies overall nuclear irregularity '
        'across the FNA sample.'),
    'texture_worst': ('Grey-scale variance (chromatin pattern)',
        'Uneven chromatin = malignant nucleus',
        'texture_worst reflects chromatin irregularity — a non-linear '
        'marker detected primarily by the DNN.'),
    'radius_worst': ('Radius of the largest cell',
        'Large radius = large cell = cancer',
        'radius_worst captures the dimensional extent of the largest cell.'),
}

print(f"\n{'='*62}")
print("  PART 2 — Clinical Meaning")
print("="*62)
for r, (idx, f, v) in enumerate(zip(top5_idx, top5_name, top5_val), 1):
    info = clinical.get(f, (f, 'High value = malignant signal',
                            f'{f} identified by ensemble SHAP.'))
    print(f"\n  Rank {r} — {f}  (|SHAP|={v:.4f})")
    print(f"    Measures: {info[0]}")
    print(f"    Meaning : {info[1]}")
    print(f"    Paper   : {info[2]}")

# ─────────────────────────────────────────────────────────────
# PART 3 — Statistical vs SHAP Comparison
# ─────────────────────────────────────────────────────────────
print(f"\n{'='*62}")
print("  PART 3 — Statistical Test vs SHAP Rank")
print("="*62)
print(f"\n  {'Feature':<26}{'Stat':>6}{'SHAP':>6}  Status")
print(f"  {'-'*52}")
for r in range(8):
    idx = int(ens_rank_idx[r]); f = FEATURE_NAMES[idx]
    sr = stat_rank.get(nm(f)); shr = r+1
    if sr is None:
        st, status = "—", "SHAP-only ⭐"
    elif abs(sr-shr) <= 2:
        st, status = str(sr), "Strong agree"
    else:
        st, status = str(sr), "Moderate"
    print(f"  {f:<26}{st:>6}{shr:>6}  {status}")

# ─────────────────────────────────────────────────────────────
# PART 4 — Malignant vs Benign Ratio
# ─────────────────────────────────────────────────────────────
print(f"\n{'='*62}")
print("  PART 4 — Top 5: Malignant vs Benign Avg Values")
print("="*62)
mal = (y_test_arr == 1); ben = (y_test_arr == 0)
print(f"\n  {'Feature':<26}{'Benign':>9}{'Malig':>9}{'Ratio':>8}  Signal")
print(f"  {'-'*62}")
for idx, f in zip(top5_idx, top5_name):
    ba = float(X_test_arr[ben, idx].mean())
    ma = float(X_test_arr[mal, idx].mean())
    ratio = ma/ba if ba > 0 else 0
    print(f"  {f:<26}{ba:>9.4f}{ma:>9.4f}{ratio:>7.2f}x  "
          f"{'HIGH in cancer' if ma>ba else 'LOW in cancer'}")

# ─────────────────────────────────────────────────────────────
# PART 5 — Model Consistency
# ─────────────────────────────────────────────────────────────
print(f"\n{'='*62}")
print("  PART 5 — Model Agreement (RF vs XGB vs DNN)")
print("="*62)
print(f"\n  {'Feature':<26}{'RF':>5}{'XGB':>5}{'DNN':>5}  Agreement")
print(f"  {'-'*52}")
agree_cnt = 0
for idx, f in zip(top5_idx, top5_name):
    rr, xr, dr = rf_rnk[idx], xgb_rnk[idx], dnn_rnk[idx]
    spread = max(rr,xr,dr) - min(rr,xr,dr)
    if spread <= 2: a = "Strong ✅"; agree_cnt += 1
    elif spread <= 5: a = "Mostly"
    else: a = "Diverge"
    print(f"  {f:<26}{rr:>5}{xr:>5}{dr:>5}  {a}")
print(f"\n  Strong agreement (spread≤2): {agree_cnt}/5")

# ─────────────────────────────────────────────────────────────
# PLOT — Top 5 Malignant vs Benign Beeswarm-style
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
np.random.seed(42)
for col, (idx, f) in enumerate(zip(top5_idx, top5_name)):
    ax = axes[col]
    bv, mv = X_test_arr[ben, idx], X_test_arr[mal, idx]
    # beeswarm-style scatter
    jb = np.random.uniform(-0.08, 0.08, len(bv))
    jm = np.random.uniform(-0.08, 0.08, len(mv))
    ax.scatter(jb, bv, c='#2980B9', s=18, alpha=0.55, label='Benign')
    ax.scatter(1+jm, mv, c='#C0392B', s=18, alpha=0.55, label='Malignant')
    # median lines
    ax.hlines(np.median(bv), -0.2, 0.2, color='navy', lw=2)
    ax.hlines(np.median(mv), 0.8, 1.2, color='darkred', lw=2)
    ax.text(0, ax.get_ylim()[1]*0.97, f'{np.median(bv):.2f}',
            ha='center', fontsize=8, color='#2980B9', fontweight='bold')
    ax.text(1, ax.get_ylim()[1]*0.97, f'{np.median(mv):.2f}',
            ha='center', fontsize=8, color='#C0392B', fontweight='bold')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Benign\n(72)','Malig\n(42)'], fontsize=8)
    ax.set_ylabel('Normalised value', fontsize=8)
    short = f.replace('_worst','\n(worst)').replace('_mean','\n(mean)')
    ax.set_title(f'Rank {col+1}\n{short}', fontsize=9, fontweight='bold')
    ax.grid(axis='y', alpha=0.25, linestyle='--')

fig.suptitle('Top 5 SHAP Features — Malignant vs Benign Distribution\n'
             'TrustBreast | WBCD', fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
for ext in ['png','pdf']:
    plt.savefig(f'Fig_Top5_Distribution.{ext}', dpi=300, bbox_inches='tight')
print(f"\n  ✅ Saved → Fig_Top5_Distribution.png / .pdf")
plt.show()

# ─────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────
print(f"\n{'='*62}")
print("  STEP 2.5 COMPLETE")
print("="*62)
print(f"""
  Top 5 Features:
    1. {top5_name[0]:<24}{top5_val[0]:.4f}
    2. {top5_name[1]:<24}{top5_val[1]:.4f}
    3. {top5_name[2]:<24}{top5_val[2]:.4f}
    4. {top5_name[3]:<24}{top5_val[3]:.4f}
    5. {top5_name[4]:<24}{top5_val[4]:.4f}  (see cross-method note)

  Key findings:
    → Top 4 = confirmed by both the statistical test and SHAP
    → texture_worst = cross-method elevated
    → All push toward MALIGNANT when HIGH

  NEXT → Step 2.6/2.7 (LIME local explanations)
""")

## STEP 7 — LIME setup + patient selection

In [ ]:
# ============================================================
# OBJECTIVE 2 — STEP 2.6
# LIME Setup — Explainer + Patient Selection
# ============================================================

try:
    import lime
    import lime.lime_tabular
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'lime', '-q'])
    import lime
    import lime.lime_tabular

import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("="*62)
print("  STEP 2.6 — LIME Setup")
print("="*62)

FEATURE_NAMES = list(X.columns)

# Training background (SMOTE-balanced) as array
X_train_arr = (X_train_sm.values if hasattr(X_train_sm, 'values')
               else np.asarray(X_train_sm))
X_test_arr  = (X_test_sc.values if hasattr(X_test_sc, 'values')
               else np.asarray(X_test_sc))

# ── Build LIME Explainer ─────────────────────────────────────
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data         = X_train_arr,
    feature_names         = FEATURE_NAMES,
    class_names           = ['Benign', 'Malignant'],
    mode                  = 'classification',
    discretize_continuous = True,
    kernel_width          = 0.75,
    random_state          = 42,
)

print("\n  ✅ LIME Explainer ready")
print(f"     Background : {X_train_arr.shape}  (SMOTE-balanced)")
print(f"     Features   : {len(FEATURE_NAMES)}")
print(f"     kernel_width : 0.75")

In [ ]:
# ── Ensemble predict_proba (LIME needs a single function) ────
def ensemble_predict_proba(X_input):
    """
    Returns (n, 2): [P(Benign), P(Malignant)]
    Soft-voting — identical to training.
    """
    X_arr = X_input.values if hasattr(X_input, 'values') else np.asarray(X_input)

    p_rf  = rf_model.predict_proba(X_arr)              # (n,2)
    p_xgb = xgb_model.predict_proba(X_arr)             # (n,2)
    p_dnn = dnn_best.predict(X_arr, verbose=0).ravel() # (n,) malignant prob
    p_dnn_full = np.column_stack([1 - p_dnn, p_dnn])   # (n,2)

    return (p_rf + p_xgb + p_dnn_full) / 3.0

# ── Sanity check ─────────────────────────────────────────────
test_p = ensemble_predict_proba(X_test_arr[:5])
print("  ✅ ensemble_predict_proba OK")
print(f"     Shape  : {test_p.shape}")
print(f"     Sum≈1? : {np.allclose(test_p.sum(axis=1), 1.0)}")
print(f"     Sample : {np.round(test_p[:3], 4)}")

In [ ]:
# ── Patient selection: highest-confidence, correctly classified ─
y_test_arr   = np.array(y_test).ravel()
ens_pred_arr = np.array(ens_pred).ravel()
prob_mal     = (np.array(prob_ensemble)[:, 1]
                if np.ndim(prob_ensemble) == 2
                else np.array(prob_ensemble).ravel())

correct = (ens_pred_arr == y_test_arr)

# 5 most-confident correct MALIGNANT
mal_pool   = np.where(correct & (y_test_arr == 1))[0]
top5_mal   = mal_pool[np.argsort(prob_mal[mal_pool])[::-1]][:5]

# 5 most-confident correct BENIGN (lowest P(Mal))
ben_pool   = np.where(correct & (y_test_arr == 0))[0]
top5_ben   = ben_pool[np.argsort(prob_mal[ben_pool])][:5]

selected_idx  = np.concatenate([top5_mal, top5_ben])
selected_lbls = [f"Patient {i+1}  -- "
                 f"{'MALIGNANT' if y_test_arr[idx]==1 else 'BENIGN'}"
                 for i, idx in enumerate(selected_idx)]

print("="*62)
print("  10 Patients Selected")
print("="*62)
print(f"\n  {'Idx':>4}{'True':>12}{'Pred':>12}{'P(Mal)':>10}")
print(f"  {'-'*40}")
for i, idx in enumerate(selected_idx):
    t = "Malignant" if y_test_arr[idx]==1 else "Benign"
    p = "Malignant" if ens_pred_arr[idx]==1 else "Benign"
    tag = "MAL" if i < 5 else "BEN"
    print(f"  {idx:>4}{t:>12}{p:>12}{prob_mal[idx]:>10.4f}  ← {tag}")

print(f"\n  Malignant idx : {top5_mal.tolist()}")
print(f"  Benign idx    : {top5_ben.tolist()}")
print(f"\n  ✅ Variables ready for Step 2.7:")
print(f"     lime_explainer, ensemble_predict_proba")
print(f"     selected_idx, selected_lbls")
print(f"     top5_mal, top5_ben")
print(f"\n  NEXT → Step 2.7 (LIME per-patient explanations)")

## STEP 7b — LIME per-patient plots (5,000 samples — for visualization only)

In [ ]:
# ============================================================
# OBJECTIVE 2 — STEP 2.7
# LIME Per-Patient Local Explanations
# 10 individual bar charts + summary grid + frequency
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

FEATURE_NAMES = list(X.columns)
y_test_arr    = np.array(y_test).ravel()
prob_arr      = (np.array(prob_ensemble)[:, 1]
                 if np.ndim(prob_ensemble) == 2
                 else np.array(prob_ensemble).ravel())
X_test_arr    = (X_test_sc.values if hasattr(X_test_sc, 'values')
                 else np.asarray(X_test_sc))

# ── Ensemble predict_proba (LIME needs a single function) ────
def ensemble_predict_proba(X_input):
    """
    Returns (n, 2): [P(Benign), P(Malignant)]
    Soft-voting — identical to training.
    """
    X_arr = X_input.values if hasattr(X_input, 'values') else np.asarray(X_input)

    p_rf  = rf_model.predict_proba(X_arr)              # (n,2)
    p_xgb = xgb_model.predict_proba(X_arr)             # (n,2)
    p_dnn = dnn_best.predict(X_arr, verbose=0).ravel() # (n,) malignant prob
    p_dnn_full = np.column_stack([1 - p_dnn, p_dnn])   # (n,2)

    return (p_rf + p_xgb + p_dnn_full) / 3.0

print("="*62)
print("  STEP 2.7 — LIME Per-Patient Explanations")
print("="*62)

# ── Verify Step 2.6 variables ────────────────────────────────
try:
    _ = lime_explainer; _ = selected_idx
    _ = selected_lbls;  _ = ensemble_predict_proba
    print(f"  Setup OK — {len(selected_idx)} patients ready")
except NameError as e:
    print(f"  ERROR: {e}  → run Step 2.6 first"); raise

# ─────────────────────────────────────────────────────────────
# PART 1 — Compute LIME (5-15 min)
# ─────────────────────────────────────────────────────────────
print("\n" + "-"*62)
print("  PART 1 — Computing LIME (5-15 min, please wait)")
print("-"*62 + "\n")

lime_results = {}
lime_probs   = {}

for i, (idx, lbl) in enumerate(zip(selected_idx, selected_lbls)):
    print(f"  [{i+1}/10] {lbl} ...", end='', flush=True)
    exp = lime_explainer.explain_instance(
        data_row     = X_test_arr[idx],
        predict_fn   = ensemble_predict_proba,
        num_features = 10,
        num_samples  = 5000,
        labels       = (1,)
    )
    lime_results[idx] = exp.as_list(label=1)
    lime_probs[idx]   = exp.predict_proba
    pb, pm = exp.predict_proba[0]*100, exp.predict_proba[1]*100
    print(f"  P(B)={pb:.1f}%  P(M)={pm:.1f}%  done")

print("\n  ✅ All 10 patients computed")

# ─────────────────────────────────────────────────────────────
# PART 2 — Text summary (6 decimals — fixed)
# ─────────────────────────────────────────────────────────────
print("\n" + "="*62)
print("  PART 2 — LIME Results (All 10)")
print("="*62)

for i, (idx, lbl) in enumerate(zip(selected_idx, selected_lbls)):
    true_c = "MALIGNANT" if y_test_arr[idx]==1 else "BENIGN"
    print(f"\n  {'='*55}")
    print(f"  Patient {i+1}  |  True: {true_c}  |  P(Mal)={prob_arr[idx]*100:.1f}%")
    print(f"  {'='*55}")
    print(f"  {'#':<3}{'Feature Condition':<36}{'Weight':>11}  Dir")
    print(f"  {'-'*55}")
    for rank, (cond, w) in enumerate(lime_results[idx], 1):
        d = "MAL" if w > 0 else "BEN"
        print(f"  {rank:<3}{cond[:35]:<36}{w:>+11.6f}  {d}")

# ─────────────────────────────────────────────────────────────
# PART 3 — Individual bar charts (10 patients)
# ─────────────────────────────────────────────────────────────
print("\n" + "-"*62)
print("  PART 3 — Individual Bar Charts")
print("-"*62)

saved = []
feat_map = {f: j for j, f in enumerate(FEATURE_NAMES)}

for i, (idx, lbl) in enumerate(zip(selected_idx, selected_lbls)):
    true_c = "MALIGNANT" if y_test_arr[idx]==1 else "BENIGN"
    raw    = lime_results[idx]
    conds  = [r[0] for r in raw]
    wts    = [float(r[1]) for r in raw]
    colors = ['#C0392B' if w > 0 else '#2980B9' for w in wts]

    plt.close('all')
    fig, ax = plt.subplots(figsize=(11, 5.5))
    y_pos = np.arange(len(conds))
    bars  = ax.barh(y_pos, wts, color=colors, edgecolor='white',
                    linewidth=0.5, height=0.65)

    for bar, w in zip(bars, wts):
        ax.text(bar.get_width() + (0.003 if w>=0 else -0.003),
                bar.get_y() + bar.get_height()/2,
                f"{w:+.4f}", va='center',
                ha='left' if w>=0 else 'right',
                fontsize=8, fontweight='bold', color='#2C3E50')

    ax.axvline(0, color='black', lw=1.0, alpha=0.6, zorder=0)
    ax.set_yticks(y_pos); ax.set_yticklabels(conds, fontsize=8.5)
    ax.set_xlabel("LIME Weight\n← BENIGN  |  MALIGNANT →",
                  fontsize=9, labelpad=6)
    ax.set_title(f"LIME Local Explanation — Patient {i+1} (True: {true_c})\n"
                 f"TrustBreast | Ensemble P(Malignant) = {prob_arr[idx]*100:.1f}%",
                 fontsize=11, fontweight='bold', pad=12)
    ax.legend(handles=[
        mpatches.Patch(color='#C0392B', label='Pushes MALIGNANT'),
        mpatches.Patch(color='#2980B9', label='Pushes BENIGN')],
        loc='lower right', fontsize=8.5, framealpha=0.85)
    ax.grid(axis='x', alpha=0.25, linestyle='--')

    plt.tight_layout()
    fn = f"LIME_Patient_{i+1:02d}_{'MAL' if true_c=='MALIGNANT' else 'BEN'}.png"
    plt.savefig(fn, dpi=300, bbox_inches='tight')
    plt.show()
    saved.append(fn)
    print(f"  ✅ {fn}")

# ─────────────────────────────────────────────────────────────
# PART 4 — 2×5 Summary Grid (FIXED tight_layout)
# ─────────────────────────────────────────────────────────────
print("\n" + "-"*62)
print("  PART 4 — 2×5 Summary Grid")
print("-"*62)

plt.close('all')
fig, axes = plt.subplots(2, 5, figsize=(28, 9))

for i, (idx, lbl) in enumerate(zip(selected_idx, selected_lbls)):
    ax = axes[0 if i < 5 else 1, i if i < 5 else i-5]
    true_c = "MALIGNANT" if y_test_arr[idx]==1 else "BENIGN"
    raw    = lime_results[idx]
    conds  = [r[0] for r in raw]
    wts    = [float(r[1]) for r in raw]
    colors = ['#C0392B' if w > 0 else '#2980B9' for w in wts]

    y_pos = np.arange(len(conds))
    ax.barh(y_pos, wts, color=colors, edgecolor='white',
            linewidth=0.4, height=0.65)
    ax.axvline(0, color='black', lw=0.8, alpha=0.5)
    ax.set_yticks(y_pos)

    short = []
    for c in conds:
        p = c.split()
        s = (p[0] if p else c).replace('concave_points','cc_pts') \
            .replace('perimeter','peri').replace('concavity','conc') \
            .replace('_worst','_W').replace('_mean','_M')
        if len(p) >= 3: s += f" {p[1]} {p[2]}"
        short.append(s[:26])
    ax.set_yticklabels(short, fontsize=6.5)
    ax.set_xlabel("LIME weight", fontsize=7)
    ax.grid(axis='x', alpha=0.2, linestyle='--')
    ax.set_title(f"P{i+1}  {true_c[:3]}  {prob_arr[idx]*100:.0f}%",
                 fontsize=8.5, fontweight='bold',
                 color='#C0392B' if true_c=='MALIGNANT' else '#2980B9', pad=5)
    ax.set_facecolor('#FFF5F5' if true_c=='MALIGNANT' else '#F0F8FF')

fig.suptitle("LIME Local Explanations — All 10 Patients\n"
             "TrustBreast Ensemble (RF+XGB+DNN) | WBCD\n"
             "Top row = Malignant (P1-P5)  |  Bottom row = Benign (P6-P10)",
             fontsize=13, fontweight='bold', y=1.02)

# ── FIX: tight_layout + subplots_adjust alag ─────────────────
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.subplots_adjust(hspace=0.45, wspace=0.55)
plt.savefig('LIME_All10_Summary_Grid.png', dpi=300, bbox_inches='tight')
plt.show()
saved.append('LIME_All10_Summary_Grid.png')
print("  ✅ LIME_All10_Summary_Grid.png")

# ─────────────────────────────────────────────────────────────
# PART 5 — Feature Frequency
# ─────────────────────────────────────────────────────────────
print("\n" + "-"*62)
print("  PART 5 — Feature Frequency Across 10 Patients")
print("-"*62)

f_mal, f_ben, f_wt = {}, {}, {}
for idx in selected_idx:
    true_c = "MALIGNANT" if y_test_arr[idx]==1 else "BENIGN"
    for cond, w in lime_results[idx]:
        matched = next((fn for fn in FEATURE_NAMES if fn in cond), None)
        if not matched: continue
        f_wt[matched]  = f_wt.get(matched, 0) + abs(float(w))
        f_mal[matched] = f_mal.get(matched, 0) + (1 if true_c=='MALIGNANT' else 0)
        f_ben[matched] = f_ben.get(matched, 0) + (1 if true_c=='BENIGN' else 0)

sorted_f = sorted(f_wt.items(), key=lambda x: x[1], reverse=True)[:15]
print(f"\n  {'Feature':<28}{'Mal':>5}{'Ben':>5}{'Total|wt|':>12}")
print(f"  {'-'*50}")
for fn, tw in sorted_f:
    print(f"  {fn:<28}{f_mal.get(fn,0):>5}{f_ben.get(fn,0):>5}{tw:>12.4f}")

# ─────────────────────────────────────────────────────────────
# PART 6 — Frequency bar chart
# ─────────────────────────────────────────────────────────────
plt.close('all')
fig, ax = plt.subplots(figsize=(12, 6))
t12 = [f[0] for f in sorted_f[:12]]
m12 = [f_mal.get(f,0) for f in t12]
b12 = [f_ben.get(f,0) for f in t12]
x, w = np.arange(len(t12)), 0.38

ax.bar(x - w/2, m12, w, label='Malignant patients (P1-P5)',
       color='#C0392B', alpha=0.85, edgecolor='white')
ax.bar(x + w/2, b12, w, label='Benign patients (P6-P10)',
       color='#2980B9', alpha=0.85, edgecolor='white')

for xi, (mv, bv) in enumerate(zip(m12, b12)):
    if mv: ax.text(xi-w/2, mv+0.05, str(mv), ha='center', fontsize=9,
                   fontweight='bold', color='#C0392B')
    if bv: ax.text(xi+w/2, bv+0.05, str(bv), ha='center', fontsize=9,
                   fontweight='bold', color='#2980B9')

short = [f.replace('concave_points','cc_pts').replace('perimeter','peri')\
          .replace('concavity','conc').replace('_worst','_W')\
          .replace('_mean','_M') for f in t12]
ax.set_xticks(x); ax.set_xticklabels(short, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Number of patients where feature appeared', fontsize=10)
ax.set_ylim(0, 6.5); ax.set_yticks(range(6))
ax.set_title("LIME Feature Frequency — How Often Each Feature Appeared\n"
             "Across 10 Patients (5 Malignant + 5 Benign) | TrustBreast | WBCD",
             fontsize=11, fontweight='bold', pad=14)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('LIME_Feature_Frequency.png', dpi=300, bbox_inches='tight')
plt.show()
saved.append('LIME_Feature_Frequency.png')
print("  ✅ LIME_Feature_Frequency.png")

# ─────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n" + "="*62)
print("  STEP 2.7 COMPLETE")
print("="*62)
print(f"\n  Most frequent LIME features (top 5):")
for j, (fn, tw) in enumerate(sorted_f[:5], 1):
    print(f"    {j}. {fn:<28} M:{f_mal.get(fn,0)}/5  B:{f_ben.get(fn,0)}/5")
print(f"\n  Files saved ({len(saved)}):")
for f in saved: print(f"    ✅ {f}")
print(f"\n  Variables ready: lime_results, lime_probs")
print(f"  NEXT → Step 2.8 (Spearman: SHAP vs LIME agreement)")


## STEP 8 — ★ Stable LIME (15,000 samples × 3 runs) — the paper's numbers come from THIS step
LIME's random generator is first reset to 42, so that every run gives the same result.

In [ ]:
from sklearn.utils import check_random_state
lime_explainer.random_state = check_random_state(42)   # Reset the RNG BACK to 42

In [ ]:
import numpy as np
from scipy.stats import spearmanr, pearsonr
import warnings
warnings.filterwarnings('ignore')

FEATURE_NAMES = list(X.columns)
N_FEAT = len(FEATURE_NAMES)
feat_map = {f: j for j, f in enumerate(FEATURE_NAMES)}
X_test_arr = (X_test_sc.values if hasattr(X_test_sc, 'values')
              else np.asarray(X_test_sc))
y_test_arr = np.array(y_test).ravel()

print("="*62)
print("  STEP 2.8 IMPROVED — Stable LIME (15000 samples × 3 runs)")
print("="*62)

NUM_SAMPLES = 15000
N_RUNS      = 3

lime_results_stable = {}

for i, idx in enumerate(selected_idx):
    lbl = "MAL" if y_test_arr[idx]==1 else "BEN"
    print(f"  [{i+1}/10] Patient {i+1} ({lbl}) — {N_RUNS} runs ...",
          end='', flush=True)

    run_matrix = np.zeros((N_RUNS, N_FEAT))
    for run in range(N_RUNS):
        exp = lime_explainer.explain_instance(
            data_row     = X_test_arr[idx],
            predict_fn   = ensemble_predict_proba,
            num_features = N_FEAT,
            num_samples  = NUM_SAMPLES,
            labels       = (1,)
        )
        for cond, weight in exp.as_list(label=1):
            matched = next((fn for fn in sorted(FEATURE_NAMES, key=len, reverse=True)
                            if fn in cond), None)
            if matched:
                run_matrix[run, feat_map[matched]] = abs(float(weight))

    lime_results_stable[idx] = run_matrix.mean(axis=0)
    print(" done")

print("\n  ✅ Stable LIME complete (averaged over 3 runs)")

In [ ]:
# Build the stable LIME matrix (10 × 30)
lime_matrix = np.zeros((len(selected_idx), N_FEAT))
for row, idx in enumerate(selected_idx):
    lime_matrix[row] = lime_results_stable[idx]

lime_global = lime_matrix.mean(axis=0)
shap_global = np.abs(shap_ensemble).mean(axis=0)

print("  ✅ Vectors ready")
print(f"     SHAP global : {shap_global.shape}")
print(f"     LIME global : {lime_global.shape}")

In [ ]:
def interpret(r):
    a = abs(r)
    if a >= 0.7: return "STRONG ✅"
    if a >= 0.5: return "MODERATE 🔶"
    if a >= 0.3: return "WEAK ⚠"
    return "POOR ❌"

# ── (A) All 30 features ──
rho_all, p_all = spearmanr(shap_global, lime_global)

# ── (B) Top-15 important features only ──
top15_idx = np.argsort(shap_global)[::-1][:15]
rho_15, p_15 = spearmanr(shap_global[top15_idx], lime_global[top15_idx])

# ── (C) Top-10 only ──
top10_idx = np.argsort(shap_global)[::-1][:10]
rho_10, p_10 = spearmanr(shap_global[top10_idx], lime_global[top10_idx])

print("="*62)
print("  Spearman — 3 Scopes")
print("="*62)
print(f"  All 30 features : ρ = {rho_all:.4f}  ({interpret(rho_all)})")
print(f"  Top 15 features : ρ = {rho_15:.4f}  ({interpret(rho_15)})")
print(f"  Top 10 features : ρ = {rho_10:.4f}  ({interpret(rho_10)})")

In [ ]:
shap_top10 = set(np.argsort(shap_global)[::-1][:10])
lime_top10 = set(np.argsort(lime_global)[::-1][:10])
overlap    = shap_top10 & lime_top10

print("="*62)
print(f"  Top-10 Overlap : {len(overlap)}/10  ({len(overlap)*10}%)")
print("="*62)
for fi in sorted(overlap, key=lambda i: -shap_global[i]):
    print(f"    ✅ {FEATURE_NAMES[fi]}")

# Per-patient (stable)
print(f"\n  {'Patient':<11}{'True':<11}{'ρ':>9}  Agreement")
print(f"  {'-'*46}")
mal_rhos, ben_rhos = [], []
for row, idx in enumerate(selected_idx):
    shap_p = np.abs(shap_ensemble[idx])
    lime_p = lime_matrix[row]
    r, _ = spearmanr(shap_p, lime_p)
    tc = "Malignant" if y_test_arr[idx]==1 else "Benign"
    (mal_rhos if y_test_arr[idx]==1 else ben_rhos).append(r)
    print(f"  P{row+1:<10}{tc:<11}{r:>9.4f}  {interpret(r)}")

mal_mean = np.nanmean(mal_rhos)
ben_mean = np.nanmean(ben_rhos)
all_mean = np.nanmean(mal_rhos + ben_rhos)
print(f"\n  Malignant mean ρ : {mal_mean:.4f}")
print(f"  Benign    mean ρ : {ben_mean:.4f}")
print(f"  Overall   mean ρ : {all_mean:.4f}")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'savefig.dpi':300, 'font.family':'DejaVu Sans', 'font.size':11})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot A — Scope comparison bar
ax = axes[0]
scopes = ['All 30', 'Top 15', 'Top 10']
rhos   = [rho_all, rho_15, rho_10]
colors = ['#95A5A6', '#3498DB', '#27AE60']
bars = ax.bar(scopes, rhos, color=colors, alpha=0.85, edgecolor='white')
for bar, r in zip(bars, rhos):
    ax.text(bar.get_x()+bar.get_width()/2, r+0.01, f'{r:.3f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(0.7, color='green', ls=':', lw=1, label='Strong (0.7)')
ax.axhline(0.5, color='orange', ls=':', lw=1, label='Moderate (0.5)')
ax.set_ylabel('Spearman ρ', fontsize=11)
ax.set_title('SHAP–LIME Agreement by Feature Scope', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.25, ls='--')

# Plot B — Malignant vs Benign per-patient
ax2 = axes[1]
colors_p = ['#C0392B' if y_test_arr[idx]==1 else '#2980B9' for idx in selected_idx]
rhos_p = [spearmanr(np.abs(shap_ensemble[idx]), lime_matrix[r])[0]
          for r, idx in enumerate(selected_idx)]
bars2 = ax2.bar(range(1,11), rhos_p, color=colors_p, alpha=0.85, edgecolor='white')
for bar, r in zip(bars2, rhos_p):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{r:.2f}',
             ha='center', va='bottom', fontsize=8, fontweight='bold')
ax2.axhline(mal_mean, color='#C0392B', ls='--', lw=1, alpha=0.6,
            label=f'Malignant avg ({mal_mean:.2f})')
ax2.axhline(ben_mean, color='#2980B9', ls='--', lw=1, alpha=0.6,
            label=f'Benign avg ({ben_mean:.2f})')
ax2.set_xticks(range(1,11)); ax2.set_xlabel('Patient', fontsize=11)
ax2.set_ylabel('Spearman ρ', fontsize=11)
ax2.set_title('Per-Patient Consistency (Stable LIME)', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 1.0); ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.25, ls='--')

plt.suptitle('Improved XAI Consistency — SHAP vs LIME (15k samples × 3 runs)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
for ext in ['png','pdf']:
    plt.savefig(f'Fig_SHAP_LIME_Consistency_Improved.{ext}', dpi=300, bbox_inches='tight')
print("  ✅ Saved → Fig_SHAP_LIME_Consistency_Improved.png / .pdf")
plt.show()

print("\n" + "="*62)
print("  IMPROVED RESULTS")
print("="*62)
print(f"""
  Global ρ (all 30) : {rho_all:.4f}  ({interpret(rho_all)})
  Top-15 ρ          : {rho_15:.4f}  ({interpret(rho_15)})
  Top-10 ρ          : {rho_10:.4f}  ({interpret(rho_10)})
  Top-10 overlap    : {len(overlap)}/10
  Malignant mean ρ  : {mal_mean:.4f}
  Benign mean ρ     : {ben_mean:.4f}
""")

In [ ]:
# ============================================================
# TABLE 3 — extracts the actual numbers (run at the end)
# Print-only; does not modify anything.
# ============================================================
import numpy as np
from scipy.stats import spearmanr

shap_order = list(np.argsort(shap_global)[::-1])
lime_order = list(np.argsort(lime_global)[::-1])
lime_global_rank = {fi: r + 1 for r, fi in enumerate(lime_order)}

top10 = shap_order[:10]
lime_within = {fi: r + 1 for r, fi in enumerate([f for f in lime_order if f in set(top10)])}

print("=" * 74)
print("  TABLE 3 — current run")
print("=" * 74)
print(f"  {'SHAP':>4}  {'Feature':<24} {'LIME rank':>10} {'LIME rank':>11}")
print(f"  {'rank':>4}  {'':<24} {'(all 30)':>10} {'(within 10)':>11}")
print("  " + "-" * 70)
for r, fi in enumerate(top10, 1):
    print(f"  {r:>4}  {FEATURE_NAMES[fi]:<24} {lime_global_rank[fi]:>10} {lime_within[fi]:>11}")

g = [lime_global_rank[fi] for fi in top10]
w = [lime_within[fi]      for fi in top10]
rho_g, p_g = spearmanr(range(1, 11), g)
rho_w, p_w = spearmanr(range(1, 11), w)

print("\n" + "=" * 74)
print(f"  rho using GLOBAL LIME ranks   : {rho_g:.4f}  (p = {p_g:.4g})")
print(f"  rho using WITHIN-10 LIME ranks: {rho_w:.4f}  (p = {p_w:.4g})")
print(f"  (code-reported top-10 rho     : {spearmanr(shap_global[top10], lime_global[top10])[0]:.4f})")
print("=" * 74)

ti = FEATURE_NAMES.index('texture_worst')
print(f"\n  texture_worst  -> SHAP rank {shap_order.index(ti)+1}, LIME rank {lime_global_rank[ti]}")
print(f"  Top-10 overlap : {len(set(top10) & set(lime_order[:10]))}/10")

print("\n  LIME's own top-10:")
for r, fi in enumerate(lime_order[:10], 1):
    mark = "  (also in SHAP top-10)" if fi in set(top10) else ""
    print(f"    {r:>2}  {FEATURE_NAMES[fi]}{mark}")


## STEP 9 — ★ RESULTS SUMMARY (share only this cell's output)

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr

shap_order = list(np.argsort(shap_global)[::-1]); lime_order = list(np.argsort(lime_global)[::-1])
lime_rank = {fi: r+1 for r, fi in enumerate(lime_order)}
top10 = shap_order[:10]
t7 = pd.DataFrame({'SHAP_rank':range(1,11), 'feature':[FEATURE_NAMES[i] for i in top10],
                   'LIME_rank':[lime_rank[i] for i in top10]})
t7.to_csv('table7_shap_lime.csv', index=False); print(t7.to_string(index=False))

rho_r, p_r = spearmanr(t7.SHAP_rank, t7.LIME_rank)
print(f"\nTable 7 rank-pair rho = {rho_r:.4f} (p = {p_r:.2e})")
for k in (30, 15, 10):
    idx = shap_order[:k]; r, pv = spearmanr(shap_global[idx], lime_global[idx])
    print(f"  scope top-{k:<2}: rho = {r:.4f} (p = {pv:.2e})")
print(f"  top-10 overlap: {len(set(top10) & set(lime_order[:10]))}/10")
print(f"  not in LIME top-10: {[FEATURE_NAMES[i] for i in top10 if i not in lime_order[:10]]}")
print(f"  LIME-only top-10  : {[FEATURE_NAMES[i] for i in lime_order[:10] if i not in top10]}")
print(f"  per-patient mean rho: malignant {mal_mean:.4f} | benign {ben_mean:.4f}")
print(f"  normalization: each model / global SHAP std  (std_rf={std_rf:.4f}, std_xgb={std_xgb:.4f}, std_dnn={std_dnn:.4f})")
print("\nFigure mapping for the paper:")
print("  Fig 6 -> Fig_Ensemble_SHAP_Beeswarm.png   (cell 17)")
print("  Fig 7 -> Fig_CrossModel_Rank_Comparison.png (cell 18)")
print("  Fig 8 -> Fig_Top5_Distribution.png        (cell 21)")

# ---------- reproducibility check: comparison with the previous run ----------
print("\n" + "="*70 + "\nREPRODUCIBILITY CHECK (vs. previous File 2 run)\n" + "="*70)
_tw_shap = shap_order.index(FEATURE_NAMES.index('texture_worst')) + 1
_tw_lime = lime_rank[FEATURE_NAMES.index('texture_worst')]
checks = [
  ("ensemble top-5", [FEATURE_NAMES[i] for i in shap_order[:5]],
   ['perimeter_worst','area_worst','concave_points_mean','texture_worst','radius_worst']),
  ("texture_worst SHAP / LIME rank", (_tw_shap, _tw_lime), (4, 4)),
  ("Table 7 LIME ranks", list(t7.LIME_rank), [2,3,1,4,5,6,7,8,13,11]),
  ("rho all-30",  round(spearmanr(shap_global, lime_global)[0], 4), 0.7259),
  ("rho top-15",  round(spearmanr(shap_global[shap_order[:15]], lime_global[shap_order[:15]])[0], 4), 0.9250),
  ("rho top-10",  round(spearmanr(shap_global[shap_order[:10]], lime_global[shap_order[:10]])[0], 4), 0.9515),
  ("top-10 overlap", len(set(top10) & set(lime_order[:10])), 8),
  ("per-patient mal / ben", (round(mal_mean,4), round(ben_mean,4)), (0.6826, 0.6502)),
]
ok_all = True
for name, now, before in checks:
    ok = (now == before); ok_all &= ok
    print(f"  {'✅' if ok else '❌'} {name:<32} now {now}   before {before}")
print("\nALL MATCH ✅ — File 2 is reproducible" if ok_all else "\nSome numbers changed ❌ — share the output for review")


## STEP 10 — Download all figures as one zip

In [ ]:
import glob, zipfile
figs = sorted(set(glob.glob('Fig_*.png') + glob.glob('*SHAP*.png') + glob.glob('LIME_*.png')))
with zipfile.ZipFile('File2_figures.zip', 'w') as z:
    for f in figs: z.write(f)
print(f"{len(figs)} figures zipped:"); [print("  ", f) for f in figs]
print("\nFor the paper:\n  Fig 6 -> Fig_Ensemble_SHAP_Beeswarm.png\n  Fig 7 -> Fig_CrossModel_Rank_Comparison.png\n  Fig 8 -> Fig_Top5_Distribution.png")
try:
    from google.colab import files; files.download('File2_figures.zip')
except Exception as e:
    print("Automatic download failed — download File2_figures.zip from the Files panel on the left.")
